# 01 · ORD delays: a first look

A first pass over the core tables built by `make dbt-build`. It covers how much data there is, when and why ORD departures run late, and what that suggests for the models. It is descriptive only: point-in-time features come later.

**Data:**
- `flight_leg`: BTS on-time data, 2015 to Aug 2026;
- `weather_obs_hourly`: IEM METARs;
- `atc_program`: FAA ATCSCC advisories;
- `calendar`.

**Caveats** (see `docs/data-catalog/`):
- **2015–2017 come from the reporting-carrier file.** It misses Envoy in 2016–2017, and it misses the regional operators that only the marketing file covers (7–16 % of ORD flights). ORD-wide comparisons therefore start in 2018.
- **2026 covers January–August only.**
- **"Late"** means the departure left at least 15 minutes after its scheduled time (BTS `DepDel15`). Cancelled flights are counted separately.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import polars as pl
from IPython.display import display

from ordfi import viz
from ordfi.settings import data_dir

viz.use_style()
pl.Config.set_tbl_rows(40)

CORE = data_dir() / "core"
con = duckdb.connect()
con.execute("set timezone = 'UTC'")
for model in ("calendar", "flight_leg", "weather_obs_hourly", "atc_program"):
    con.execute(f"create view {model} as select * from read_parquet('{CORE / model}.parquet')")


def sql(query: str) -> pl.DataFrame:
    return con.sql(query).pl()

## 1 · How much data, and where coverage changes

Counts are flights in the BTS files, not ORD's total operations. International, cargo and general-aviation movements are not in BTS. Two breaks are coverage changes, not traffic changes.

In [ ]:
monthly = sql("""
    select cast(date_trunc('month', flight_date) as date) as month, count(*) as flights
    from flight_leg
    group by all
    order by month
""")

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(monthly["month"], monthly["flights"], color=viz.CATEGORICAL[0])
ax.set_ylim(0, monthly["flights"].max() * 1.15)
viz.thousands_axis(ax)
viz.titles(
    ax, "ORD flights per month in the BTS data", "Departures and arrivals, Jan 2015 – Aug 2026"
)
notes = {  # date: (label, label offset in points)
    "2016-01-01": ("Envoy missing\n(2016–2017)", (0, -46)),
    "2018-01-01": ("Marketing-carrier file\nadds regional operators", (-80, 42)),
    "2020-04-01": ("COVID-19", (0, -46)),
}
for when, (text, offset) in notes.items():
    point = monthly.filter(pl.col("month") == pl.lit(when).str.to_date())
    ax.annotate(
        text,
        xy=(point["month"][0], point["flights"][0]),
        xytext=offset,
        textcoords="offset points",
        ha="center",
        fontsize=9,
        color=viz.TEXT_SECONDARY,
        arrowprops={"arrowstyle": "-", "color": viz.MUTED, "linewidth": 0.8},
    )
plt.show()

monthly.group_by(pl.col("month").dt.year().alias("year")).agg(
    pl.col("flights").sum(), pl.len().alias("months")
).sort("year")

## 2 · How often departures run late

Late departures fell to 10 % in 2020 and have risen every year since. 2025 and Jan–Aug 2026 are the worst of the period. Delays also got longer: the 90th-percentile delay went from about 50 min before COVID to over 70 min in 2026.

In [ ]:
yearly = sql("""
    select
        year,
        count(*) as departures,
        100 * avg(dep_del15::int) filter (where not cancelled) as late_pct,
        100 * avg(cancelled::int) as cancelled_pct,
        median(dep_delay) filter (where not cancelled) as median_delay_min,
        quantile_cont(dep_delay, 0.9) filter (where not cancelled) as p90_delay_min
    from flight_leg
    where direction = 'departure'
    group by all
    order by year
""")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
panels = [("late_pct", "Departures 15+ min late"), ("cancelled_pct", "Departures cancelled")]
for ax, (column, title) in zip(axes, panels, strict=True):
    ax.bar(yearly["year"], yearly[column], width=0.6, color=viz.CATEGORICAL[0])
    ax.set_xticks(yearly["year"], [f"’{year % 100:02d}" for year in yearly["year"]])
    ax.grid(axis="x", visible=False)
    viz.percent_axis(ax)
    viz.titles(ax, title, "Share of ORD departures by year (2026: Jan–Aug)")
    for row in (yearly.sort(column).row(0, named=True), yearly.sort(column).row(-1, named=True)):
        ax.annotate(
            f"{row[column]:.1f} %",
            xy=(row["year"], row[column]),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            fontsize=9,
            color=viz.TEXT_SECONDARY,
        )
plt.show()

yearly.with_columns(pl.col("late_pct", "cancelled_pct").round(1))

## 3 · When: season × time of day

Delays build through the day. Departures scheduled before 7 am are late about 10 % of the time; on summer evenings it is 40 % or more. This is the thunderstorm season meeting the end of the day, when earlier delays have piled up.

In [ ]:
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
cells = sql("""
    select
        month(flight_date) as month,
        cast(left(crs_dep_time, 2) as integer) % 24 as hour,
        count(*) as departures,
        100 * avg(dep_del15::int) filter (where not cancelled) as late_pct
    from flight_leg
    where direction = 'departure' and year between 2018 and 2025 and year != 2020
    group by all
""").filter(pl.col("departures") >= 500)
grid = cells.sort("hour").pivot(on="hour", index="month", values="late_pct").sort("month")
hours = [int(column) for column in grid.columns[1:]]

fig, ax = plt.subplots(figsize=(10, 4.4))
mesh = ax.pcolormesh(
    grid.drop("month").to_numpy(),
    cmap=viz.SEQUENTIAL,
    vmin=0,
    vmax=50,
    edgecolors=viz.SURFACE,
    linewidth=1.5,
)
ax.set_xticks([i + 0.5 for i in range(len(hours))], [f"{hour:02d}" for hour in hours])
ax.set_yticks([i + 0.5 for i in range(12)], MONTHS)
ax.invert_yaxis()
ax.grid(False)
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_xlabel("Scheduled departure hour (local)")
scale = fig.colorbar(mesh, ax=ax, shrink=0.85, pad=0.02)
scale.outline.set_visible(False)
scale.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0f} %"))
scale.ax.tick_params(length=0, labelcolor=viz.TEXT_SECONDARY)
viz.titles(
    ax,
    "Late departures by month and scheduled hour",
    "Share 15+ min late, ORD departures 2018–2025 excluding 2020; cells with ≥ 500 departures",
)
plt.show()

grid.with_columns(pl.col(grid.columns[1:]).round(0))

## 4 · Why: weather at ORD

Each departure is matched to the ORD weather in its scheduled hour (`weather_obs_hourly`: the worst flight category reported in that hour). Thunderstorm and snow hours are shown separately whatever their category.

| Category | Ceiling (ft) | | Visibility (mi) |
|---|---|---|---|
| VFR | > 3,000 | and | > 5 |
| MVFR | 1,000–3,000 | or | 3–5 |
| IFR | 500–999 | or | 1–3 |
| LIFR | < 500 | or | < 1 |

Thunderstorms nearly triple the late share and cut capacity for hours. Low ceilings matter much less than storms. Note that this is weather *at the time*: a model can only use forecasts.

In [ ]:
CONDITIONS = ["VFR", "MVFR", "IFR", "LIFR", "Snow", "Thunderstorm"]
weather = (
    sql("""
    with departures as (
        select date_trunc('hour', sched_dep_utc) as hour, dep_del15, cancelled
        from flight_leg
        where direction = 'departure' and year >= 2018
    )
    select
        case
            when w.any_thunder then 'Thunderstorm'
            when w.any_snow then 'Snow'
            else w.worst_flight_category
        end as condition,
        count(*) as departures,
        100 * avg(d.dep_del15::int) filter (where not d.cancelled) as late_pct,
        100 * avg(d.cancelled::int) as cancelled_pct
    from departures as d
    inner join weather_obs_hourly as w on w.valid_time = d.hour
    group by condition
    having condition is not null
""")
    .with_columns(pl.col("condition").cast(pl.Enum(CONDITIONS)))
    .sort("condition")
)

fig, ax = plt.subplots(figsize=(8.5, 3.6))
positions = range(len(weather))
ax.barh(positions, weather["late_pct"], height=0.5, color=viz.CATEGORICAL[0])
ax.set_yticks(positions, weather["condition"].cast(pl.String))
ax.invert_yaxis()
ax.grid(axis="x", visible=True)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
viz.percent_axis(ax, axis="x")
ax.set_xlim(0, weather["late_pct"].max() * 1.12)  # room for tip labels
for position, value in zip(positions, weather["late_pct"], strict=True):
    ax.annotate(
        f"{value:.0f} %",
        xy=(value, position),
        xytext=(4, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color=viz.TEXT_SECONDARY,
    )
viz.titles(
    ax,
    "Late departures by ORD weather in the scheduled hour",
    "Share 15+ min late, ORD departures 2018 – Aug 2026",
)
plt.show()

weather.with_columns(pl.col("late_pct", "cancelled_pct").round(1))

## 5 · Why: FAA traffic-management programs

Ground delay programs (GDPs) and ground stops at ORD hold inbound flights at their origins. Below:
- **Program days:** the local dates on which an ORD program period starts, from `atc_program`, excluding proposals and cancellations.
- **The table:** departures scheduled while an ORD program was in effect, compared with the rest.

Programs regulate *arrivals*, so for departures this is a proxy for "ORD under traffic management". Departures scheduled during an ORD program were late 57 % of the time and cancelled 15 % of the time, against 18 % and 1.5 % otherwise (2018 – Aug 2026).

In [ ]:
programs = sql("""
    select
        year(timezone('America/Chicago', start_utc)) as year,
        count(distinct cast(timezone('America/Chicago', start_utc) as date))
            filter (where program_type = 'gdp') as gdp_days,
        count(distinct cast(timezone('America/Chicago', start_utc) as date))
            filter (where program_type = 'ground_stop') as ground_stop_days
    from atc_program
    where airport = 'ORD' and not is_cancellation and not is_proposed and start_utc is not null
    group by all
    order by year
""")

fig, ax = plt.subplots(figsize=(10, 3.8))
width, gap = 0.36, 0.04
series = [("gdp_days", "Ground delay program"), ("ground_stop_days", "Ground stop")]
for slot, (column, label) in enumerate(series):
    offset = (width + gap) / 2 * (1 if slot else -1)
    ax.bar(
        programs["year"] + offset,
        programs[column],
        width=width,
        color=viz.CATEGORICAL[slot],
        label=label,
    )
ax.set_xticks(programs["year"], [f"’{year % 100:02d}" for year in programs["year"]])
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left", ncols=2)
ax.set_ylim(0, max(programs["gdp_days"].max(), programs["ground_stop_days"].max()) * 1.25)
viz.titles(
    ax,
    "Days with an ORD ground delay program or ground stop",
    "Local dates on which an ORD program period starts (2026: through Oct 26)",
)
plt.show()

in_program = sql("""
    with spans as (
        select start_utc, end_utc
        from atc_program
        where airport = 'ORD' and not is_cancellation and not is_proposed
            and start_utc is not null and end_utc is not null
    )
    select
        exists (
            select 1 from spans
            where f.sched_dep_utc >= spans.start_utc and f.sched_dep_utc < spans.end_utc
        ) as during_ord_program,
        count(*) as departures,
        100 * avg(f.dep_del15::int) filter (where not f.cancelled) as late_pct,
        100 * avg(f.cancelled::int) as cancelled_pct
    from flight_leg as f
    where f.direction = 'departure' and f.year >= 2018
    group by all
    order by during_ord_program
""")
display(programs)
in_program.with_columns(pl.col("late_pct", "cancelled_pct").round(1))

## 6 · Why: the inbound aircraft

Most ORD departures are flown by an aircraft that has just arrived at ORD. Each departure is paired with the same tail number's latest scheduled ORD arrival before it, keeping scheduled turns of 20–240 minutes (about 78 % of 2018+ departures).

How late the inbound aircraft is predicts the departure better than anything above. In historical data, though, the tail is the *actual* aircraft, so live features must use the currently assigned one.

In [ ]:
turns = sql("""
    with arrivals as (
        select tail_number, sched_arr_utc as sched_in, arr_delay
        from flight_leg
        where direction = 'arrival' and not cancelled and not diverted and year >= 2018
    ),
    departures as (
        select tail_number, sched_dep_utc, dep_del15
        from flight_leg
        where direction = 'departure' and not cancelled and year >= 2018
    ),
    paired as (
        select
            d.dep_del15,
            a.arr_delay as inbound_delay,
            date_diff('minute', a.sched_in, d.sched_dep_utc) as turn_min
        from departures as d
        asof join arrivals as a
            on d.tail_number = a.tail_number and d.sched_dep_utc > a.sched_in
    )
    select
        least(greatest(floor(inbound_delay / 15) * 15, -15), 180)::integer as inbound_bin,
        count(*) as departures,
        100 * avg(dep_del15::int) as late_pct
    from paired
    where turn_min between 20 and 240 and inbound_delay is not null
    group by all
    order by inbound_bin
""")


def bin_label(start: int) -> str:
    return "Early" if start < 0 else "180+" if start >= 180 else f"{start}–{start + 14}"


labels = [bin_label(start) for start in turns["inbound_bin"]]

fig, ax = plt.subplots(figsize=(10, 3.8))
positions = list(range(len(turns)))
ax.plot(
    positions,
    turns["late_pct"],
    color=viz.CATEGORICAL[0],
    marker="o",
    markersize=6,
    markeredgecolor=viz.SURFACE,
    markeredgewidth=2,
)
ax.set_xticks(positions, labels)
ax.set_xlabel("Inbound aircraft's arrival delay at ORD (minutes)")
ax.set_ylim(0, 100)
viz.percent_axis(ax)
for index in (0, len(turns) - 1):
    ax.annotate(
        f"{turns['late_pct'][index]:.0f} %",
        xy=(index, turns["late_pct"][index]),
        xytext=(0, 9),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        color=viz.TEXT_SECONDARY,
    )
viz.titles(
    ax,
    "Departures 15+ min late by how late the inbound aircraft was",
    "ORD departures 2018 – Aug 2026, scheduled 20–240 min after the same tail's arrival",
)
plt.show()

turns.with_columns(pl.Series("inbound_delay_min", labels), pl.col("late_pct").round(1))

## 7 · Who: operating carriers

These are the ten largest operating carriers at ORD in 2023–2025 (regional operators fly for United, American and Delta). Differences partly reflect schedules and routes, not only performance.

In [ ]:
NAMES = {
    "UA": "United",
    "OO": "SkyWest",
    "AA": "American",
    "MQ": "Envoy",
    "YX": "Republic",
    "ZW": "Air Wisconsin",
    "G7": "GoJet",
    "DL": "Delta",
    "NK": "Spirit",
    "WN": "Southwest",
    "9E": "Endeavor",
    "AS": "Alaska",
    "B6": "JetBlue",
    "F9": "Frontier",
    "PT": "Piedmont",
    "OH": "PSA",
}
carriers = sql("""
    select
        operating_carrier as carrier,
        count(*) as departures,
        100 * avg(dep_del15::int) filter (where not cancelled) as late_pct,
        100 * avg(cancelled::int) as cancelled_pct
    from flight_leg
    where direction = 'departure' and year between 2023 and 2025
    group by all
    order by departures desc
    limit 10
""").sort("late_pct", descending=True)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
positions = range(len(carriers))
ax.barh(positions, carriers["late_pct"], height=0.5, color=viz.CATEGORICAL[0])
ax.set_yticks(positions, [f"{NAMES.get(c, c)} ({c})" for c in carriers["carrier"]])
ax.invert_yaxis()
ax.grid(axis="x", visible=True)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
viz.percent_axis(ax, axis="x")
ax.set_xlim(0, carriers["late_pct"].max() * 1.12)  # room for tip labels
for position, value in zip(positions, carriers["late_pct"], strict=True):
    ax.annotate(
        f"{value:.0f} %",
        xy=(value, position),
        xytext=(4, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color=viz.TEXT_SECONDARY,
    )
viz.titles(
    ax,
    "Late departures by operating carrier",
    "Share 15+ min late, ten largest operators at ORD, 2023–2025",
)
plt.show()

carriers.with_columns(pl.col("late_pct", "cancelled_pct").round(1))

## 8 · What this means for the models

**Labels.**
- Use `dep_del15` and `arr_del15`, plus delay minutes for quantile models.
- Train ORD-wide models on 2018 onward: earlier years miss carriers.
- BTS cause codes change for flights from 2027-01-01.

**Signals worth building as point-in-time features:**
1. **Inbound aircraft delay.** It is the strongest single signal. It is known only once the inbound flight is airborne or landed, so it matters most at short horizons and needs live data.
2. **Time of day × season.** These are known in advance and cheap: they form a baseline every model should beat.
3. **Thunderstorms and snow at ORD.** Live models can only use these through forecasts: TAF (needs our own bulletin parser), NBM and LAMP.
4. **FAA programs at ORD.** Their effect is large. Planning advisories and live NAS Status make them partly predictable a few hours ahead.

**Carriers and holidays.**
- Carriers differ, but carrier identity mixes in schedule and route effects; model it with care.
- Holidays barely move the late share in these data. Cancellations do jump on the holidays themselves.

**Next:**
- program in-effect intervals;
- hourly TAF features;
- an `aircraft_rotation` core model;
- an as-of feature builder with a no-leakage property test.

In [ ]:
con.close()